# NB-1 / 22SJ-3 ?? ? ????? ??????????

???????????????

1. `deg_pro/??.csv` ?????????????
2. ?????8?????
3. `weather_region_daily_minimal.csv` ? `?? + ??` ???
4. ?????????????
5. ??? `deg_pro/output` ?CSV??

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', 100)

In [ ]:
# ????

def ????????????(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'Kaggle' / 'data').exists():
            return p
    raise FileNotFoundError('Kaggle/data ????????')

cwd = Path.cwd()
project_root = ????????????(cwd)

deg_dir = project_root / 'Kaggle' / 'data' / 'deg_pro'
if not deg_dir.exists():
    raise FileNotFoundError(f'deg_pro ????????????: {deg_dir}')

sales_candidates = sorted([p for p in deg_dir.glob('*.csv')], key=lambda p: p.stat().st_size, reverse=True)
if not sales_candidates:
    raise FileNotFoundError(f'deg_pro ??CSV??????: {deg_dir}')

sales_path = sales_candidates[0]

weather_candidates = list((project_root / 'Kaggle' / 'data').rglob('weather_region_daily_minimal.csv'))
if not weather_candidates:
    raise FileNotFoundError('weather_region_daily_minimal.csv ????????')
weather_path = weather_candidates[0]

print('??CSV:', sales_path)
print('??CSV:', weather_path)

In [ ]:
# ??CSV???????????8??
raw = pd.read_csv(sales_path, encoding='utf-8-sig', header=None)
if raw.shape[1] < 8:
    raise ValueError(f'????????: {raw.shape[1]} ?')

raw = raw.iloc[:, :8].copy()
raw.columns = [
    '???_raw',
    '??_raw',
    '?????_raw',
    '??',
    'JAN???',
    '???',
    '??????',
    '??????',
]

sales = raw.copy()
sales['???'] = pd.to_datetime(sales['???_raw'].astype(str), format='%Y%m%d', errors='coerce')
sales['??????'] = pd.to_numeric(sales['??????'], errors='coerce')
sales['??????'] = pd.to_numeric(sales['??????'], errors='coerce')

sales = sales.dropna(subset=['???', '??', 'JAN???'])
sales = sales[sales['??'].isin(['NB-1', '22SJ-3'])].copy()

print('????:', len(sales))
print('????:')
print(sales['??'].value_counts())

In [ ]:
# ???? -> 8??
???? = ['???', '??', '??', '??', '??', '??', '??', '????']

????????? = {
    '???': '???',
    '??': '??', '??': '??', '??': '??', '??': '??', '??': '??', '??': '??',
    '??': '??', '??': '??', '??': '??', '??': '??', '??': '??', '??': '??', '???': '??',
    '??': '??', '??': '??', '??': '??', '??': '??', '??': '??', '??': '??',
    '??': '??', '??': '??', '??': '??', '??': '??',
    '??': '??', '??': '??', '??': '??', '??': '??', '??': '??', '???': '??',
    '??': '??', '??': '??', '??': '??', '??': '??', '??': '??',
    '??': '??', '??': '??', '??': '??', '??': '??',
    '??': '????', '??': '????', '??': '????', '??': '????',
    '??': '????', '??': '????', '???': '????', '??': '????',
}

sales['?????_???'] = (
    sales['?????_raw']
    .astype(str)
    .str.strip()
    .str.replace(r'[????]', '', regex=True)
)

sales['??'] = sales['?????_???'].map(?????????)

# ????_raw?8?????????????
??raw_??? = sales['??_raw'].astype(str).str.strip()
???? = sales['??'].isna() & ??raw_???.isin(????)
sales.loc[????, '??'] = ??raw_???[????]

sales['??8?????'] = sales['??'].isin(????)
print('??8??????????:', int(sales['??8?????'].sum()), '/', len(sales))

print('
????????????????10??:')
print(sales.loc[~sales['??8?????'], '?????_raw'].astype(str).value_counts().head(10))

sales_jp = sales[sales['??8?????']].copy()

In [ ]:
# ???????? ?????
sales_daily = (
    sales_jp.groupby(['???', '??', '??'], as_index=False)
    .agg(
        ??????=('??????', 'sum'),
        ??????=('??????', 'sum'),
        JAN?=('JAN???', 'nunique'),
        ????=('JAN???', 'size'),
    )
    .sort_values(['???', '??', '??'])
)

print('??????:', len(sales_daily))
print('????:', sales_daily['???'].min(), '->', sales_daily['???'].max())
display(sales_daily.head())

In [ ]:
# ????????????
weather = pd.read_csv(weather_path, encoding='utf-8-sig')
weather['??'] = pd.to_datetime(weather['date'], errors='coerce')
weather = weather.dropna(subset=['??', 'region']).copy()

weather = weather.rename(columns={
    'region': '??',
    'temp_avg_c': '????',
    'temp_max_c': '????',
    'temp_min_c': '????',
    'precip_mm': '???',
    'sunshine_h': '????',
    'snowfall_cm': '???',
    'humidity_pct': '????',
    'station_count': '???',
})

for c in ['????', '????', '????', '???', '????', '???', '????', '???']:
    weather[c] = pd.to_numeric(weather[c], errors='coerce')

print('????:', len(weather))
print('????:', weather['??'].min(), '->', weather['??'].max())
display(weather.head())

In [ ]:
# ???????????+???
merged = sales_daily.merge(
    weather,
    left_on=['???', '??'],
    right_on=['??', '??'],
    how='inner'
)

print('?????:', len(merged))
print('?????:', merged['???'].min(), '->', merged['???'].max())
print('?????:')
print(merged['??'].value_counts())
display(merged.head())

In [ ]:
# ?????????? vs ?????
??? = ['????', '????', '????', '???', '????', '???', '????']

rows = []
for ??, g in merged.groupby('??'):
    for ?? in ???:
        rows.append({
            '??': ??,
            '??': ??,
            '????': g['??????'].corr(g[??]),
            '?????': len(g),
        })

corr_model = pd.DataFrame(rows).sort_values(['??', '????'], ascending=[True, False])
display(corr_model)

In [ ]:
# ??????????????30???
rows = []
for (??, ??), g in merged.groupby(['??', '??']):
    if len(g) < 30:
        continue
    for ?? in ???:
        rows.append({
            '??': ??,
            '??': ??,
            '??': ??,
            '????': g['??????'].corr(g[??]),
            '?????': len(g),
        })

corr_model_region = pd.DataFrame(rows)
if not corr_model_region.empty:
    corr_model_region = corr_model_region.sort_values(['??', '??', '????'], ascending=[True, True, False])

display(corr_model_region.head(30))

In [ ]:
# ???????????????
monthly = (
    merged.assign(?=lambda d: d['???'].dt.to_period('M').dt.to_timestamp())
    .groupby(['?', '??'], as_index=False)
    .agg(
        ??????=('??????', 'sum'),
        ????=('????', 'mean')
    )
)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for i, ?? in enumerate(sorted(monthly['??'].unique())):
    m = monthly[monthly['??'] == ??]
    ax1 = axes[i]
    ax2 = ax1.twinx()

    ax1.plot(m['?'], m['??????'], color='tab:blue', label='??????')
    ax2.plot(m['?'], m['????'], color='tab:red', alpha=0.7, label='????')

    ax1.set_title(f'{??} ????????? / ?????')
    ax1.set_ylabel('????')
    ax2.set_ylabel('????(?)')

plt.tight_layout()
plt.show()

In [ ]:
# CSV????????????
out_dir = deg_dir / 'output'
out_dir.mkdir(parents=True, exist_ok=True)

???? = out_dir / '??_??_?????_NB1_22SJ3.csv'
????_?? = out_dir / '??_??_??_???_NB1_22SJ3.csv'
????_???? = out_dir / '??_??_??_?????_NB1_22SJ3.csv'

merged.to_csv(????, index=False, encoding='utf-8-sig')
corr_model.to_csv(????_??, index=False, encoding='utf-8-sig')
if isinstance(corr_model_region, pd.DataFrame) and not corr_model_region.empty:
    corr_model_region.to_csv(????_????, index=False, encoding='utf-8-sig')

print('??:', ????)
print('??:', ????_??)
if ????_????.exists():
    print('??:', ????_????)

## ???

- ???CSV?? `?? / ?? / ?? / ?? / ??` ??????????????????8?????????????
- ????????Pearson?????????????????????